In [ ]:
!pip install transformers accelerate torch pandas openpyxl

In [ ]:
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

# Set the model name
model_name = "Qwen/Qwen2.5-3B-Instruct"

print("Loading the model and tokenizer... Please wait.")

# Load the tokenizer to convert text into a format the model understands
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load the model directly onto the GPU (device_map="auto") for faster processing
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)
print("Model loaded successfully!")

Loading the model and tokenizer... Please wait.


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Model loaded successfully!


In [ ]:
# Load the original Excel file
input_file = "EcowattTips_raw.xlsx"
df = pd.read_excel(input_file)

def generate_advice(raw_tip):
    # Return an empty string if the tip is missing or empty
    if pd.isna(raw_tip):
        return ""

    # Define the prompt to instruct the LLM on how to format the advice
    prompt = f"""
    You are a friendly energy-saving assistant for the EcoWatt app, helping households across all regions of Saudi Arabia save electricity.
    Rewrite the following energy-saving tip to make it:
    1. Highly actionable (tell the user exactly what to do).
    2. Simple and user-friendly.
    3. Strictly ONE single sentence.
    4. Maximum 20 words.
    6. Avoid repetition and unnatural phrasing.

    Original raw tip: "{raw_tip}"

    Friendly and Actionable Advice:
    """

    # Format the prompt using the model's chat template
    messages = [
        {"role": "system", "content": "You are a helpful, concise, and friendly assistant."},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Move the formatted text to the GPU for processing
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    # Generate the advice (limited to 40 tokens for brevity)
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=40,
        do_sample=False
    )

    # Extract only the newly generated text (ignoring the input prompt)
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    # Clean up the final output (remove extra spaces and quotes)
    return response.strip().replace('"', '')

print("Data loaded successfully and function is ready.")

Data loaded successfully and function is ready.


In [ ]:
print("Generating advice for all tips...")

# Enable progress bar for pandas apply function
tqdm.pandas()

# Apply the function to each row in the 'raw tips' column and store in a new 'advice' column
df['advice'] = df['raw tips'].progress_apply(generate_advice)

# Overwrite the original Excel file with the updated dataset
output_file = "EcowattTips.xlsx"
df.to_excel(output_file, index=False)

print(f"Process completed successfully! The file '{output_file}' has been updated and overwritten.")

Generating advice for all tips...


100%|██████████| 53/53 [01:02<00:00,  1.18s/it]

Process completed successfully! The file 'EcowattTips.xlsx' has been updated and overwritten.
